[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Tasks](Tasks.md) | [Task 1](Task-count-addresses.md) | Notebook ⮕ | [Slides](slides/ETP-Week-02-BGP.pptx)

### [Network Infrastructure Data Science (NIDS) Assignment]

---

# How origin ASes announce IPv4 prefixes in BGP

In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [ ]:
%pip install pybgpkit-parser pelicanfs pytricia pandas

import json
import gzip
import urllib.request
import bz2
import io
import statistics
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

import ipaddress
import pytricia

import pandas as pd
import matplotlib.pyplot as plt
import pybgpkit_parser as bgpkit
from pelicanfs.core import OSDFFileSystem


@contextmanager
def open_safe(filename, encoding="utf-8"):
    path = Path(filename)
    suffix = path.suffix.lower()
    if suffix == ".gz":
        with gzip.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".bz2":
        with bz2.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".zip":
        with zipfile.ZipFile(path) as zf:
            with zf.open(zf.namelist()[0]) as raw:
                yield io.TextIOWrapper(raw, encoding=encoding)
    else:
        with path.open(encoding=encoding) as f:
            yield f

---

## Task 1: CCDF of Origin AS IPv4 Prefix and Address Counts

In this task you will:

1. Fetch a BGP routing table (RIB) snapshot from a RouteViews collector via OSDF.
2. Build a `prefix → set of origin ASes` mapping and identify MOAS prefixes (announced by more than one AS).
3. For single-origin IPv4 prefixes, compute each AS's **prefix count** and **address count**.
   The address count uses a Pytricia prefix trie with longest-prefix match to avoid
   double-counting nested prefixes — see the [Task 1 guide](Task-count-addresses.md).
4. Plot a **Complementary Cumulative Distribution Function (CCDF)** with two curves:
   - by prefix count (bottom x-axis)
   - by address count (top x-axis)

Both axes are log-scaled. Each point `(x, y)` means *y* ASes have at least *x* prefixes (or addresses).

In [ ]:
OSDF_BASE = "https://osdf-director.osg-htc.org"

collectors = ["route-views3"]


def prefix2as_for_collector(collector):
    osdf = OSDFFileSystem()
    path = f"/routeviews/{collector}/bgpdata/2026.05/RIBS"
    objects = sorted(osdf.ls(path), key=lambda x: x["name"])
    url = OSDF_BASE + objects[0]["name"]
    print(f"  reading {url}")

    prefix_origins = defaultdict(set)
    i = 0
    for elem in bgpkit.Parser(url=url):
        # YOUR CODE HERE
        # Add elem.origin_asns to prefix_origins[elem.prefix].
        pass
    return prefix_origins

collector = collectors[0]
print(f"processing {collector}...")
prefix_origins = prefix2as_for_collector(collector)

In [ ]:
def count_addresses_per_asn(ipv4_single_origin):
    pyt = pytricia.PyTricia(32)
    # YOUR CODE HERE 
    # Count unique IPv4 addresses per origin AS from a
    #   BGP RIB snapshot of (prefix, asn) pairs.
    # No double-counting: each address is credited to exactly one AS.
    # Longest-prefix match: the more-specific prefix (/N higher) wins.
    # Pytricia trie: pyt.children(p) returns all nested prefixes;
    #   pyt.parent(c) returns the immediately enclosing prefix.
    # Count = raw size minus direct children's sizes only (not all
    #   descendants) to prevent grandchildren being subtracted twice.


moas_count = 0
asn_prefix_count = defaultdict(int)
asn_to_nets = defaultdict(list)
ipv4_single_origin = []  # (net, asn) for IPv4 single-origin prefixes
for prefix, origins in prefix_origins.items():
    net = ipaddress.ip_network(prefix, strict=False)
    if net.version != 4:
        continue
    # YOUR CODE HERE
    # If len(origins) > 1, this is a MOAS prefix: increment moas_count.
    # Otherwise extract the single origin: asn = next(iter(origins))
    #   - increment asn_prefix_count[asn]
    #   - append net to asn_to_nets[asn]
    #   - append (str(net), asn) to ipv4_single_origin

asn_address_count = count_addresses_per_asn(ipv4_single_origin)

total_prefixes = len(prefix_origins)
single_count = total_prefixes - moas_count
single_pct = 100.0 * single_count / total_prefixes
moas_pct = 100.0 * moas_count / total_prefixes
counts_list = list(asn_prefix_count.values())
avg = sum(counts_list) / len(counts_list)
median = statistics.median(counts_list)

print(f"total prefixes:")
print(f"   total :          {total_prefixes}")
print(f"   moas:            {moas_count} ({moas_pct:.1f}%)")
print(f"   single asn:      {single_count} ({single_pct:.1f}%)")
print(f"origin (number of prefixes)")
print(f"   minimum:         {min(counts_list)}")
print(f"   maximum:         {max(counts_list)}")
print(f"   average per ASN: {avg:.1f}")
print(f"   median per ASN:  {int(median)}")

total_addr = sum(asn_address_count.values())
print(f"total IPv4 addresses attributed: {total_addr}")

In [ ]:
count_dist = Counter(counts_list)
# YOUR CODE HERE
# Compute CCDF arrays for prefix counts.
# Using count_dist and asn_prefix_count:
#   x_prefix: sorted list of unique prefix count values
#   total_asns: total number of origin ASes
#   y_asns: list where y_asns[i] = number of ASes with at least x_prefix[i] prefixes.
#     Hint: start remaining = total_asns, subtract count_dist[xi] at each step.
pass

# YOUR CODE HERE
# Compute CCDF arrays for address counts.
# Using asn_address_count (ASN -> address count, built in the cell above):
#   addr_counts_list: address counts from asn_address_count.values(), excluding zeros
#   addr_count_dist: Counter of addr_counts_list
#   x_addr: sorted list of unique address count values
#   y_addr: list where y_addr[i] = number of ASes with at least x_addr[i] addresses
pass

fig, ax1 = plt.subplots()
ax1.plot(x_prefix, y_asns, marker=".", markersize=5, color="C0", label="by prefix count (bottom axis)")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Number of IPv4 prefixes (bottom axis)")
ax1.set_ylabel("Number of ASNs")

ax2 = ax1.twiny()
ax2.plot(x_addr, y_addr, marker=".", markersize=5, color="C1", label="by address count")
ax2.set_xscale("log")
ax2.set_xlabel("Number of IPv4 addresses")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
ax1.set_title(f"CCDF: IPv4 prefix and address count per AS ({collector})")
ax1.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

# Capture origin series for the combined plot
origin_x_prefix, origin_y = x_prefix, y_asns
origin_x_addr, origin_y_addr = x_addr, y_addr

In [ ]:
# --- Summary statistics for Task 1 (origin AS) ---
# Printed here so the values referenced in the questions below are explicit.
origin_prefix_vals = list(asn_prefix_count.values())
origin_addr_vals = [v for v in asn_address_count.values() if v > 0]
total_prefix_sum = sum(origin_prefix_vals)
total_addr_sum = sum(origin_addr_vals)

print("Origin AS prefix counts:")
print(f"   ASes:            {len(origin_prefix_vals)}")
print(f"   min / max:       {min(origin_prefix_vals)} / {max(origin_prefix_vals)} "
      f"(largest AS = {100.0*max(origin_prefix_vals)/total_prefix_sum:.1f}% of all single-origin prefixes)")
print(f"   mean / median:   {sum(origin_prefix_vals)/len(origin_prefix_vals):.1f} / {int(statistics.median(origin_prefix_vals))}")
print(f"   median AS:       {100.0*statistics.median(origin_prefix_vals)/total_prefix_sum:.4f}% of all prefixes")
print("Origin AS address counts:")
print(f"   min / max:       {min(origin_addr_vals)} / {max(origin_addr_vals)} "
      f"(largest AS = {100.0*max(origin_addr_vals)/total_addr_sum:.1f}% of all attributed addresses)")
print(f"   total attributed:{total_addr_sum}")
print("MOAS:")
print(f"   total prefixes:  {len(prefix_origins)}")
print(f"   moas / pct:      {moas_count} ({100.0*moas_count/len(prefix_origins):.1f}%)")
print(f"   single-origin:   {len(prefix_origins)-moas_count} ({100.0*(len(prefix_origins)-moas_count)/len(prefix_origins):.1f}%)")

### Question 1

What does the shape of the CCDF reveal about how IPv4 prefixes are distributed among origin ASes?

YOUR ANSWER HERE

### Question 2

What percentage of IPv4 prefixes are MOAS (announced by more than one origin AS)? What does MOAS represent, and why does it matter for routing security?

YOUR ANSWER HERE

### Question 3

Why do the prefix-count curve and the address-count curve diverge on the plot? What does this tell you about how address space is allocated relative to prefix announcements?

YOUR ANSWER HERE

---

## Task 2: CCDF of Customer Cone IPv4 Prefix and Address Counts

In this task you will:

1. Load the CAIDA PPDC customer-cone file, which maps each root AS to the set of ASes in its customer cone.
2. Compute each root AS's **cone prefix count** and **cone address count** by aggregating counts across every cone member (reusing `asn_prefix_count` and `asn_to_nets` from Task 1).
3. Plot a **CCDF** with two curves on a log-log scale:
   - by cone prefix count (bottom x-axis)
   - by cone address count (top x-axis)

A prefix is counted toward a customer cone once per cone; the same prefix may appear in multiple cones.

In [ ]:
AS_CONE_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as-relationships/20260501.ppdc-ases.txt.bz2"
AS_CONE_PATH = Path("data/20260501.ppdc-ases.txt.bz2")

print(f"Downloading {AS_CONE_URL} ...")
AS_CONE_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS_CONE_URL, AS_CONE_PATH)
if not AS_CONE_PATH.exists(): 
    print (f"Unable to save file {AS_CONE_PATH}")
    exit() 

print("loading customer cone data...")
asn_to_cone = {}
with open_safe(AS_CONE_PATH) as fin:
    for line in fin:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        # YOUR CODE HERE
        # Split the line into parts. The first token is the root ASN;
        # the remaining tokens are all ASNs in the cone (including the root).
        # Store as: asn_to_cone[int(parts[0])] = {int(p) for p in parts}

# YOUR CODE HERE
# Build cone_prefix_count as a dict mapping each root_asn to the sum of
# asn_prefix_count.get(m, 0) for every m in cone_members.
cone_prefix_count = {}

cone_counts = [v for v in cone_prefix_count.values() if v > 0]
cone_avg = sum(cone_counts) / len(cone_counts) if cone_counts else 0.0
cone_median = statistics.median(cone_counts) if cone_counts else 0

print(f"cone (number of prefixes)")
print(f"   minimum:         {min(cone_counts) if cone_counts else 0}")
print(f"   maximum:         {max(cone_counts) if cone_counts else 0}")
print(f"   average per ASN: {cone_avg:.1f}")
print(f"   median per ASN:  {int(cone_median)}")

# YOUR CODE HERE
# Build cone_address_count. For each root_asn:
#   Collect all nets from asn_to_nets for every member of its cone,
#   sort by (network_address, prefixlen).
#   Walk the sorted list and sum only non-overlapping address ranges using
#   a running covered_end pointer: add net.num_addresses when
#   net_start > covered_end, then update covered_end.
cone_address_count = {}

In [ ]:
# YOUR CODE HERE
# Compute CCDF arrays for cone prefix counts.
# Using cone_counts (list of per-root-AS cone prefix counts, built above):
#   cone_prefix_dist: Counter of cone_counts
#   x_cone: sorted list of unique cone prefix count values
#   y_cone: list where y_cone[i] = number of ASes with at least x_cone[i] prefixes in their cone.
#     Hint: start remaining = len(cone_counts), subtract cone_prefix_dist[xi] at each step.
pass

# YOUR CODE HERE
# Compute CCDF arrays for cone address counts.
# Using cone_address_count (root ASN -> address count, built above):
#   cone_addr_list: address counts from cone_address_count.values(), excluding zeros
#   cone_addr_dist: Counter of cone_addr_list
#   x_addr: sorted list of unique address count values
#   y_addr: list where y_addr[i] = number of ASes with at least x_addr[i] addresses in their cone
pass

fig, ax1 = plt.subplots()
ax1.plot(x_cone, y_cone, marker=".", markersize=5, color="C0", label="by prefix count")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Number of IPv4 prefixes in customer cone")
ax1.set_ylabel("Number of ASNs")

ax2 = ax1.twiny()
ax2.plot(x_addr, y_addr, marker=".", markersize=5, color="C1", label="by address count")
ax2.set_xscale("log")
ax2.set_xlabel("Number of IPv4 addresses in customer cone")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
ax1.set_title(f"CCDF: customer cone IPv4 prefix and address count per AS ({collector})")
ax1.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

# Capture cone series for the combined plot
cone_x_prefix, cone_y = x_cone, y_cone
cone_x_addr, cone_y_addr = x_addr, y_addr

In [ ]:
# --- Summary statistics for Task 2 (customer cone) ---
# Printed here so the values referenced in the questions below are explicit.
cone_prefix_vals = [v for v in cone_prefix_count.values() if v > 0]
cone_addr_vals = [v for v in cone_address_count.values() if v > 0]

# Global routing-table totals for context (cones overlap, so these are reference points, not sums)
global_prefixes = sum(asn_prefix_count.values())
global_addrs = sum(v for v in asn_address_count.values() if v > 0)

print("Customer cone prefix counts:")
print(f"   ASes:            {len(cone_prefix_vals)}")
print(f"   min / max:       {min(cone_prefix_vals)} / {max(cone_prefix_vals)} "
      f"(largest cone covers {100.0*max(cone_prefix_vals)/global_prefixes:.1f}% of all routed prefixes)")
print(f"   mean / median:   {sum(cone_prefix_vals)/len(cone_prefix_vals):.1f} / {int(statistics.median(cone_prefix_vals))}")
print(f"   median cone:     {100.0*statistics.median(cone_prefix_vals)/global_prefixes:.4f}% of all routed prefixes")
print("Customer cone address counts:")
print(f"   min / max:       {min(cone_addr_vals)} / {max(cone_addr_vals)} "
      f"(largest cone covers {100.0*max(cone_addr_vals)/global_addrs:.1f}% of all attributed addresses)")
print(f"   mean / median:   {sum(cone_addr_vals)/len(cone_addr_vals):.1f} / {int(statistics.median(cone_addr_vals))}")

### Combined view: origin vs. customer cone

The plot below overlays all four CCDFs computed above — origin prefix count, origin address count, cone prefix count, and cone address count — on a single pair of log-log axes (prefixes on the bottom axis, addresses on the top). Use it to answer Questions 4 and 5.

In [ ]:
# --- Combined CCDF: origin vs customer cone, prefix and address counts ---
# Reuses the four series already computed above; no data is regenerated.
fig, ax1 = plt.subplots()
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("Number of IPv4 prefixes")
ax1.set_ylabel("Number of ASNs")

ax1.plot(origin_x_prefix, origin_y, marker=".", markersize=4, color="C0", label="origin prefix count")
ax1.plot(cone_x_prefix, cone_y, marker=".", markersize=4, color="C2", label="cone prefix count")

ax2 = ax1.twiny()
ax2.set_xscale("log")
ax2.set_xlabel("Number of IPv4 addresses")
ax2.plot(origin_x_addr, origin_y_addr, marker=".", markersize=4, color="C1", label="origin address count")
ax2.plot(cone_x_addr, cone_y_addr, marker=".", markersize=4, color="C3", label="cone address count")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
ax1.set_title(f"CCDF: origin vs customer cone, prefix and address counts ({collector})")
ax1.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

### Question 4

What does the shape of the customer cone CCDF reveal about how IPv4 reachability is distributed across ASes?

YOUR ANSWER HERE

### Question 5

Why are the customer-cone values (both prefix count and address count) larger than the origin values? Reference the combined plot and the printed statistics.

YOUR ANSWER HERE

## Load ASN's organization's name and country

Download the AS2Org file and build a mapping from each ASN to its organization name and country code.

In [ ]:
AS2ORG_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as2org/as2org.jsonl"
AS2ORG_PATH = Path("data/as2org.jsonl")

if not AS2ORG_PATH.exists():
    print(f"Downloading {AS2ORG_URL} ...")
    AS2ORG_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(AS2ORG_URL, AS2ORG_PATH)
if not AS2ORG_PATH.exists():
    print(f"Unable to save file {AS2ORG_PATH}")
    exit()

asn_to_info = {}
with open_safe(AS2ORG_PATH) as fin:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        country = rec.get("country", "")
        name = rec.get("orgName", "")
        for asn in rec.get("members", []):
            asn_to_info[int(asn)] = {"country": country, "name": name}
print(f"num asn's with information: {len(asn_to_info.keys())}")

## Task 3: Ranked Table of Top ASNs

Build a ranked table of ASNs that appear in the **top 3 of any** of the four size metrics below. For each qualifying ASN, show its rank and raw count for all four metrics, along with the organization name and country from AS2Org.

**Ranking:** For each metric, rank all ASNs by descending value (rank 1 = highest value). An ASN qualifies for the table if its rank is ≤ 3 in at least one metric. Sort rows so ASNs with the best rank in any column appear first.

| Column | Description |
|---|---|
| ASN | Autonomous System Number |
| name | Organization name (from AS2Org) |
| country | Country code (from AS2Org) |
| o prefix rank | Rank by own prefix count (1 = most prefixes announced) |
| o prefix total | Number of own prefixes |
| o addr rank | Rank by own IPv4 address count (1 = most addresses) |
| o addr total | Number of own IPv4 addresses |
| c prefix rank | Rank by customer cone prefix count (1 = largest cone) |
| c prefix total | Number of prefixes in customer cone |
| c addr rank | Rank by customer cone address count |
| c addr total | Number of IPv4 addresses in customer cone |

In [ ]:
# YOUR CODE HERE
# 1. Compute a 1-based rank dict for each of the four metrics:
#      asn_prefix_count, asn_address_count, cone_prefix_count, cone_address_count
#    Rank 1 = highest value. (Hint: sort items by value descending, then enumerate.)
#
# 2. Collect the set of ASNs that appear in the top 3 of ANY of those four rank dicts.
#
# 3. For each qualifying ASN, look up name and country from asn_to_info.
#    Build a list of row dicts with keys:
#      ASN, name, country,
#      o prefix rank, o prefix total,
#      o addr rank,   o addr total,
#      c prefix rank, c prefix total,
#      c addr rank,   c addr total
#
# 4. Rank so that the top ASN has the highest combination rank across all four metrics
#    so a [1,1,3,4] would beat [1,2,2,3], which would beat [2,2,3,9999], which would beat [3,3,3,3].
pass

### Question 6

Write a single sentence for each of the top 4 organizations: what are they, and why would they be ranked so high?

YOUR ANSWER HERE